# EXP15 — 표현 수리(전처리 번들) 위의 activation expert system (Colab)

사전 등록: `manuscript/report/0821.md §6`. **원노브 = feature 표현** (4항목 번들 고지):
(a) TCP_FLAGS 3종 → 비트 24개, (b) 준-상수 7개 제거, (c) placeholder 0→NaN, (d) heavy-tail 28개 log1p → 46→60 features.
방법·그룹핑은 exp12c와 동일, global 비교열은 같은 run·같은 표현으로 재-fit.

⚠ **고RAM 런타임 필수** (suite pkl 로드에 순간 ~25GB), GPU는 **VRAM ≥ 20GB** (L4/A100; T4 16GB 불가 — global 1M 단계 실측 ~16.6GiB).
⚠ GPU가 로컬 4090과 다르므로 exp12c(기록 run)와의 비교에는 §12n 재현성 밴드(클래스 F1 ±0.01)를 적용. 판정의 핵심 비교(system_prep vs global_prep)는 같은 run 안이라 영향 없음.


## 1. Drive 마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 경로
Drive 구조 (exp4 때 만든 그대로):
```
MyDrive/imbal_cic_tabpfn/
  src/            nfv3_v3_common.py, exp_utils.py, *.ckpt  (+ 이번에 nfv3_v3_exp15_prep_system.py 업로드)
  data/           nfv3_energy_suite_uncapped_scenarios.pkl  (14.7GB — exp4 때 업로드했다면 그대로)
  results/        <- --out-root
```


In [ ]:
import os, glob, shutil, sys

DRIVE_ROOT = '/content/drive/MyDrive/imbal_cic_tabpfn'
CODE_DIR   = DRIVE_ROOT + '/src'
DATA_DIR   = DRIVE_ROOT + '/data'
OUT_ROOT   = DRIVE_ROOT + '/results'
WORK    = '/content/work'
SCRATCH = DRIVE_ROOT + '/tabpfn_cache'
MODELS_DIR = DRIVE_ROOT + '/saved_models'

for d in (SCRATCH, OUT_ROOT, MODELS_DIR, WORK + '/exp', WORK + '/scripts'):
    os.makedirs(d, exist_ok=True)

for label, path in [('CODE_DIR', CODE_DIR), ('DATA_DIR', DATA_DIR)]:
    ok = os.path.isdir(path)
    print(f'{label:10s} {path}   exists={ok}')
    if not ok:
        raise FileNotFoundError(path + ' 가 없습니다. DRIVE_ROOT를 확인하세요.')

PKL = os.path.join(DATA_DIR, 'nfv3_energy_suite_uncapped_scenarios.pkl')
if not os.path.isfile(PKL):
    raise FileNotFoundError(
        PKL + ' 가 없습니다.\n'
        'exp15(cic2018 uncapped)는 이 pkl이 필요합니다 — 로컬 data/에서 업로드하세요 (14.7GB).')
print(f'suite pkl  {os.path.getsize(PKL)/1e9:.1f} GB  OK')


## 3. 패키지 설치 — **fork 필수** (실패하면 여기서 멈춥니다)
공개 pip `tabpfn==8.2.0`에는 v3 class-attention 내부(`_project_qk`)가 없어서
게이트 featurize가 `decoder not found`로 죽습니다 (로컬 fork가 릴리스보다 앞선 코드).
로컬 `imbalcic/tabpfn/tabpfn_fork_for_colab.zip` (735KB)를 Drive `src/`에 업로드해 두세요.
이미 이 세션에서 pip tabpfn을 import한 적이 있으면 **런타임 재시작 후 처음부터** 실행하세요.


In [ ]:
import glob, os, shutil, subprocess, sys, importlib

if 'tabpfn' in sys.modules:
    raise RuntimeError('이 커널에 이미 (구버전) tabpfn이 import돼 있습니다 — '
                       '런타임 다시 시작 후 셀을 처음부터 실행하세요.')

ZIP_NAME = 'tabpfn_fork_for_colab.zip'
hits = glob.glob(os.path.join(CODE_DIR, '**', ZIP_NAME), recursive=True)
if not hits:
    raise FileNotFoundError(
        f'{CODE_DIR} 아래에 {ZIP_NAME} 이 없습니다 — '
        f'로컬 imbalcic/tabpfn/{ZIP_NAME} 를 Drive src/에 업로드하세요.')

FORK = '/content/tabpfn_fork'
shutil.rmtree(FORK, ignore_errors=True)
shutil.unpack_archive(hits[0], FORK)
root = FORK if os.path.isfile(os.path.join(FORK, 'pyproject.toml')) \
    else os.path.dirname(glob.glob(FORK + '/**/pyproject.toml', recursive=True)[0])

for args in (['uninstall', '-y', '-q', 'tabpfn'], ['install', '-q', root]):
    r = subprocess.run([sys.executable, '-m', 'pip', *args],
                       capture_output=True, text=True)
    if r.returncode != 0 and args[0] == 'install':
        print(r.stdout[-4000:]); print(r.stderr[-4000:])
        raise RuntimeError('fork 설치 실패 — 위 로그를 보세요')

importlib.invalidate_caches()
import tabpfn
pkg_dir = os.path.dirname(tabpfn.__file__)
have = subprocess.run(['grep', '-rl', '_project_qk',
                       os.path.join(pkg_dir, 'architectures')],
                      capture_output=True, text=True).stdout.strip()
if not have:
    raise RuntimeError('설치된 tabpfn에 _project_qk가 없습니다 — '
                       'pip 캐시가 남았을 수 있으니 런타임 재시작 후 다시 실행하세요.')
print('tabpfn(fork):', tabpfn.__version__, '\n  ', pkg_dir, '\n  v3 class-attention OK')


## 4. GPU·RAM 확인 — **T4 또는 저RAM이면 여기서 멈춥니다**


In [ ]:
import torch, psutil
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
ram = psutil.virtual_memory().total / 1e9
print(f'GPU: {name}  VRAM {vram:.0f} GB   RAM {ram:.0f} GB')
if vram < 20:
    raise RuntimeError('VRAM < 20GB — global 1M 단계(실측 ~16.6GiB)가 OOM. L4/A100으로 바꾸세요.')
if ram < 40:
    raise RuntimeError('RAM < 40GB — suite pkl 로드 불가. 고RAM 런타임으로 바꾸세요.')


## 5. 코드 배치
`src/` 아래에서 3개 파일을 찾아 로컬 작업 트리에 놓습니다.
**이번에 새로 올릴 파일: `nfv3_v3_exp15_prep_system.py`** (나머지 둘은 exp4 때 이미 있음).


In [ ]:
SCRIPT = 'nfv3_v3_exp15_prep_system.py'
NEEDED = ['nfv3_v3_common.py', SCRIPT, 'exp_utils.py']

missing = []
for name in NEEDED:
    hits = glob.glob(os.path.join(CODE_DIR, '**', name), recursive=True)
    if not hits:
        missing.append(name); continue
    dst = (WORK + '/scripts/' if name == 'exp_utils.py' else WORK + '/exp/') + name
    shutil.copy(hits[0], dst)
    print(f'  {name:36s} <- {hits[0]}')

if missing:
    raise FileNotFoundError(
        f'{CODE_DIR} 아래에서 못 찾은 파일: {missing}\n'
        '로컬 repo의 tabpfn/nfv3_v3_exp15_prep_system.py 를 src/에 업로드하세요.')
print('\nOK')


## 6. 체크포인트
`src/` 안의 `.ckpt`를 찾습니다. 없으면 HF에서 받아 Drive에 캐시합니다.


In [ ]:
CKPT_NAME = 'tabpfn-v3-classifier-v3_20260417_multiclass.ckpt'
hits = glob.glob(os.path.join(CODE_DIR, '**', CKPT_NAME), recursive=True)
if hits:
    CKPT = hits[0]
else:
    from huggingface_hub import hf_hub_download
    src = hf_hub_download(repo_id='Prior-Labs/tabpfn_3', filename=CKPT_NAME)
    CKPT = os.path.join(CODE_DIR, CKPT_NAME)
    shutil.copy(src, CKPT)
print(CKPT, f'{os.path.getsize(CKPT)/1e6:.0f} MB')


## 7. 실행 설정 — 사전 등록값 그대로, 바꾸지 마세요 (원노브)


In [ ]:
TARGET     = 'cic2018'
FIT_MODE   = 'fit_with_cache'   # 0820.md §7: cache 모드만 1M×4M 가능
TEST_BATCH = 500_000
GLOBAL_CTX = 1_000_000          # global 비교열: 자연 비례 1M, 같은 run·같은 표현


## 8. 실행 (~1.5-2.5h; 로그가 그대로 흐릅니다)


In [ ]:
cmd = [
    sys.executable, SCRIPT,
    '--target-dataset', TARGET,
    '--fit-mode', FIT_MODE,
    '--test-batch-size', str(TEST_BATCH),
    '--fit-global',
    '--global-context-size', str(GLOBAL_CTX),
    '--data-dir', DATA_DIR,
    '--out-root', OUT_ROOT,
    '--model-path', CKPT,
    '--resume-dir', SCRATCH,
    '--models-dir', MODELS_DIR,
]
print(' '.join(cmd))


In [ ]:
import subprocess

def run(command):
    p = subprocess.Popen(command, cwd=WORK + '/exp', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ})
    captured = []
    for line in p.stdout:
        sys.stdout.write(line)
        captured.append(line)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(
            f'실행 실패 (exit code {p.returncode}) — 위 로그를 보세요. '
            '아래 결과 셀은 실행하지 마세요(예전 run을 읽게 됩니다).')
    out = [l.split('Wrote ', 1)[1].strip() for l in captured if l.startswith('Wrote ')]
    if not out:
        raise RuntimeError("'Wrote <dir>' 줄이 없습니다 — 아티팩트가 안 만들어졌습니다.")
    return out[-1]

RUN_DIR = run(cmd)
print('\nrun dir:', RUN_DIR)


## 9. 결과 — exp12c(원시 표현, 기록 run 20260821_070437)와 나란히


In [ ]:
import pandas as pd

t = pd.read_csv(os.path.join(RUN_DIR, 'per_class_metrics.csv'))
piv = t.pivot(index='class', columns='method', values='f1').round(4)

# exp12c 기록 (원시 46-feature, 로컬 4090 — 비교엔 ±0.01 재현성 밴드 적용)
rec = pd.DataFrame({
    'exp12c_system_raw': {'benign': .9741, 'bot': .9963, 'brute_force': .9159,
        'ddos': .9939, 'dos': .7805, 'infiltration': .1178, 'web_attacks': .0647,
        'macro_avg': .6919},
    'exp12c_global_raw': {'benign': .9918, 'bot': .9984, 'brute_force': .9159,
        'ddos': .9627, 'dos': .7827, 'infiltration': .0968, 'web_attacks': .1601,
        'macro_avg': .7012}})
print(piv.join(rec).to_string())

act = pd.read_csv(os.path.join(RUN_DIR, '4a_full_activation_counts.csv'))
p = act.pivot_table(index='class', columns='winner', values='rows', aggfunc='sum').fillna(0).astype(int)
share = (p.T / p.sum(axis=1)).T.round(4)
print('\n=== activation share (행=true class) ===')
print(share.to_string())
print('\n기록 발화율(exp12c): benign .9575 ddos .9894 dos .6412 bot .9970 brute .9998 inf .2972 web .8585')


## 판정 규칙 (0821.md §6, 결과 전 고정)
- ① 같은 run: `system_prep` vs `global_prep_reference` — 표현 위에서 시스템이 global을 넘는가
- ② 표현 효과 분해: exp15 두 열 vs exp12c 두 열 (±0.01 밴드)
- 관심 지표: inf 발화율 0.297→? · benign 오발화 4.23%→? · web precision 0.034→? · ddos 0.994 유지
- 상한 인지: inf test의 71%(동일-벡터 충돌+benign-쌍둥이)는 표현 수리로 원리상 불가
